In [1]:
import os
from pathlib import Path

# set the root directory as the current working directory
os.chdir(Path.cwd().parent)
print(f"Current working directory: {os.getcwd()}")

Current working directory: /shared-docker/CyberSec-Reasoner


In [2]:
import torch
import random
import logging
import warnings

%load_ext autoreload
%autoreload 2

from datasets import DatasetDict
from src.utils.logger import setup_logging
from src.config_loader import load_config
from src.utils.utils import setup_env, log_gpu_info
from src.utils.seed import setup_seed
from src.utils.wandb import init_wandb, finish_wandb
from src.model_loader import load_tokenizer, load_qlora_base_model
from src.data_loader import load_dataset, apply_chat_tempalte
from src.dataset_stats import print_token_stats
from src.trainer import build_sft_config, build_trainer, save_adapter, merge_and_save

warnings.filterwarnings("ignore")
logger = logging.getLogger(__name__)

In [3]:
if torch.cuda.is_available():
    print(f"Number of available GPUs: {torch.cuda.device_count()}")
    print(f"GPU Name: {torch.cuda.get_device_name()}")
    print(f"Total GPU Memory: {torch.cuda.get_device_properties().total_memory / 1024**3:.2f} GB")
else:
    print("No GPU detected! Using CPU...")

Number of available GPUs: 1
GPU Name: 
Total GPU Memory: 191.69 GB


*****
# Training

In [4]:
## setup logging
setup_logging()

config_path = "./src/configs/rsft_qwen3.5_4b_cold_start.yaml"
model_cfg, dataset_cfg, lora_config, training_cfg, wandb_cfg, paths_cfg, _ = load_config(config_path)

# setup environment
setup_env()

# log GPU info
log_gpu_info()

# setup seed
setup_seed(training_cfg)

# init wandb
wandb_run = init_wandb(training_cfg["report_to"], wandb_cfg)

2026-04-12 14:12:14 | INFO     | src.utils.logger | Logging is set up.
2026-04-12 14:12:14 | INFO     | src.config_loader | Config loaded from: ./src/configs/rsft_qwen3.5_4b_cold_start.yaml
2026-04-12 14:12:14 | INFO     | src.config_loader | Output directories created (if they did not exist)
2026-04-12 14:12:14 | INFO     | src.config_loader | Model configs: {'name': 'Qwen/Qwen3.5-4B', 'torch_dtype': 'bfloat16', 'load_in_8bit': True, 'llm_int8_threshold': 6.0, 'llm_int_skip_modules': 'none', 'llm_int8_enable_fp32_cpu_offload': True, 'attn_implementation': 'flash_attention_2', 'device_map': 'cuda:0', 'trust_remote_code': True}
2026-04-12 14:12:14 | INFO     | src.config_loader | Dataset configs: {'path': './data/processed/primus_reasoning_cwe_mapping_dataset', 'max_length': 4096, 'column': 'messages', 'padding_side': 'right', 'truncation_side': 'left', 'packing': False}
2026-04-12 14:12:14 | INFO     | src.config_loader | LoRA configs: {'r': 64, 'lora_alpha': 128, 'lora_dropout': 0.05,

In [5]:
# tokenizer
tokenizer = load_tokenizer(model_cfg, dataset_cfg)

2026-04-12 14:12:20 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/Qwen/Qwen3.5-4B/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-04-12 14:12:20 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen3.5-4B/851bf6e806efd8d0a36b00ddf55e13ccb7b8cd0a/config.json "HTTP/1.1 200 OK"
2026-04-12 14:12:20 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/Qwen/Qwen3.5-4B/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
2026-04-12 14:12:20 | WARNING  | huggingface_hub.utils._http | Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
2026-04-12 14:12:20 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen3.5-4B/851bf6e806efd8d0a36b00ddf55e13ccb7b8cd0a/tokenizer_config.json "HTTP/1.1 200 OK"
2026-04-12 14:12:20 | INFO     | httpx | HTTP Request: HEAD https://huggi

In [14]:
# modify the chat template of tokenizer to calculate the assistant only loss
tokenizer.chat_template = """
{% for message in messages %}
{% if message['role'] == 'system' %}
<|im_start|>system
{{ message['content'] }}<|im_end|>
{% elif message['role'] == 'user' %}
<|im_start|>user
{{ message['content'] }}<|im_end|>
{% elif message['role'] == 'assistant' %}
<|im_start|>assistant
{% generation %}
{{ message['content'] }}
{% endgeneration %}
<|im_end|>
{% endif %}
{% endfor %}
"""

In [15]:
# dataset
dataset = load_dataset(dataset_cfg)

# take 1000 of the dataset for cold-start reasoining sft
cold_start = DatasetDict({
    "train": dataset["train"].select(range(1000)).shuffle(seed=42)
})

dataset = cold_start

# apply chat template
dataset = apply_chat_tempalte(dataset, tokenizer, "qwen")
print_token_stats(dataset, tokenizer)

2026-04-12 14:24:12 | INFO     | src.data_loader | Dataset loaded from data/processed/primus_reasoning_cwe_mapping_dataset with 2307 training, 0 validation, 0 test samples.
2026-04-12 14:24:12 | INFO     | src.data_loader | Applying chat template for model family: qwen


Dataset: DatasetDict({
    train: Dataset({
        features: ['messages'],
        num_rows: 2307
    })
})


Applying Qwen chat template:   0%|          | 0/10 [00:00<?, ? examples/s]

2026-04-12 14:24:12 | INFO     | src.data_loader | Chat template applied to dataset.
Processing train: 100% 10/10 [00:00<00:00, 653.99it/s]
2026-04-12 14:24:12 | INFO     | src.dataset_stats | [train] stats -> Min: 783 | Avg: 976.70 | Max: 1569


In [17]:
print(dataset["train"][random.randint(0, (len(dataset["train"]) -1))]["text"])


<|im_start|>system
You are a cybersecurity reasoning expert specialized in vulnerability analysis and classification.

Your task is to analyze CVE (Common Vulnerabilities and Exposures) descriptions and map them to the most appropriate CWE (Common Weakness Enumeration).

You must follow a structured reasoning workflow:

1. Understand the vulnerability context and affected components  
2. Identify the vulnerability type (e.g., XSS, memory corruption, improper validation)  
3. Determine the root cause of the weakness  
4. Map the root cause to the most appropriate CWE category  

Guidelines:
- Focus on root cause rather than surface-level keywords  
- Use precise cybersecurity terminology  
- Ensure reasoning clearly supports the final CWE selection  
- Prefer the most specific applicable CWE when possible  

Avoid:
- Guessing without justification  
- Contradictions between reasoning and conclusion  
- Irrelevant or overly generic explanations  
- Blindly copying CWE identifiers from t

In [11]:
# load model
log_gpu_info("Before loading model")
model = load_qlora_base_model(model_cfg)
log_gpu_info("After loading model")

2026-04-12 14:14:08 | INFO     | src.utils.utils | GPU 0: , 205.82 GB VRAM. Before loading model
2026-04-12 14:14:08 | INFO     | src.model_loader | Loading QLoRA model: Qwen/Qwen3.5-4B with bitsandbytes config: BitsAndBytesConfig {
  "_load_in_4bit": false,
  "_load_in_8bit": true,
  "bnb_4bit_compute_dtype": "float32",
  "bnb_4bit_quant_storage": "uint8",
  "bnb_4bit_quant_type": "fp4",
  "bnb_4bit_use_double_quant": false,
  "llm_int8_enable_fp32_cpu_offload": true,
  "llm_int8_has_fp16_weight": false,
  "llm_int8_skip_modules": null,
  "llm_int8_threshold": 6.0,
  "load_in_4bit": false,
  "load_in_8bit": true,
  "quant_method": "bitsandbytes"
}

2026-04-12 14:14:08 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/Qwen/Qwen3.5-4B/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-04-12 14:14:08 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen3.5-4B/851bf6e806efd8d0a36b00ddf55e13ccb7b8cd0a/config.json "HTTP/1

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/426 [00:00<?, ?it/s]

2026-04-12 14:14:15 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/Qwen/Qwen3.5-4B/resolve/main/generation_config.json "HTTP/1.1 404 Not Found"
2026-04-12 14:14:15 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/Qwen/Qwen3.5-4B/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-04-12 14:14:15 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen3.5-4B/851bf6e806efd8d0a36b00ddf55e13ccb7b8cd0a/config.json "HTTP/1.1 200 OK"
2026-04-12 14:14:16 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/Qwen/Qwen3.5-4B/resolve/main/custom_generate/generate.py "HTTP/1.1 404 Not Found"
2026-04-12 14:14:16 | INFO     | src.model_loader | QLoRA model Qwen/Qwen3.5-4B loaded with 4,205,751,296 parameters.
2026-04-12 14:14:16 | INFO     | src.model_loader | Model Loaded on device: cuda:0
2026-04-12 14:14:16 | INFO     | src.utils.utils | GPU 0: , 205.82 GB VRAM. After loading model


In [18]:
# init trainer
sft_config = build_sft_config(dataset_cfg, training_cfg, wandb_cfg["run_name"], evaluation=False)

trainer = build_trainer(
    model, 
    tokenizer, 
    dataset,
    sft_config,
    lora_config,
    evaluation=False
)

2026-04-12 14:25:28 | WARNING  | src.trainer | Evaluation is disabled. Skipping evaluation during training
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.
2026-04-12 14:25:28 | INFO     | src.trainer | Create LoRA Configs...
2026-04-12 14:25:28 | INFO     | src.model_loader | LoRA config created with r=64, alpha=128, dropout=0.05, bias=none, task_type=CAUSAL_LM, use_rslora=True,
target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'], 
2026-04-12 14:25:28 | INFO     | src.trainer | Prepare the model for LoRA fine-tuning...
2026-04-12 14:25:29 | INFO     | src.trainer | Peft Model Info: None
2026-04-12 14:25:29 | INFO     | src.trainer | Innitializing the SFTTrainer...


trainable params: 84,934,656 || all params: 4,290,685,952 || trainable%: 1.9795


Tokenizing train dataset:   0%|          | 0/10 [00:00<?, ? examples/s]

2026-04-12 14:25:29 | INFO     | src.trainer | SFTTrainer initialized successfully.
2026-04-12 14:25:29 | INFO     | src.trainer | Number of trainable parameters    : 84,934,656
2026-04-12 14:25:29 | INFO     | src.trainer | Number of non-trainable parameters: 4,205,751,296
2026-04-12 14:25:29 | INFO     | src.trainer | Effective batch size (per_device * grad_accum): 8
2026-04-12 14:25:29 | INFO     | src.trainer | Steps per epoch                   : 2
2026-04-12 14:25:29 | INFO     | src.trainer | Total training steps              : 2


In [23]:
print(trainer.train_dataset[0])

{'messages': [{'content': 'You are a cybersecurity reasoning expert specialized in vulnerability analysis and classification.\n\nYour task is to analyze CVE (Common Vulnerabilities and Exposures) descriptions and map them to the most appropriate CWE (Common Weakness Enumeration).\n\nYou must follow a structured reasoning workflow:\n\n1. Understand the vulnerability context and affected components  \n2. Identify the vulnerability type (e.g., XSS, memory corruption, improper validation)  \n3. Determine the root cause of the weakness  \n4. Map the root cause to the most appropriate CWE category  \n\nGuidelines:\n- Focus on root cause rather than surface-level keywords  \n- Use precise cybersecurity terminology  \n- Ensure reasoning clearly supports the final CWE selection  \n- Prefer the most specific applicable CWE when possible  \n\nAvoid:\n- Guessing without justification  \n- Contradictions between reasoning and conclusion  \n- Irrelevant or overly generic explanations  \n- Blindly co

In [24]:
# start training
log_gpu_info("Before training")
logger.info("Starting training...")
result = trainer.train()
log_gpu_info("After training")
logger.info(f"Training completed. Training result: {result}")

2026-04-12 14:28:04 | INFO     | src.utils.utils | GPU 0: , 205.82 GB VRAM. Before training
2026-04-12 14:28:04 | INFO     | __main__ | Starting training...
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 248046, 'pad_token_id': 248044}.


Step,Training Loss


2026-04-12 14:28:20 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/Qwen/Qwen3.5-4B/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-04-12 14:28:20 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen3.5-4B/851bf6e806efd8d0a36b00ddf55e13ccb7b8cd0a/config.json "HTTP/1.1 200 OK"
2026-04-12 14:28:20 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/Qwen/Qwen3.5-4B/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-04-12 14:28:20 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen3.5-4B/851bf6e806efd8d0a36b00ddf55e13ccb7b8cd0a/config.json "HTTP/1.1 200 OK"
2026-04-12 14:28:22 | INFO     | src.utils.utils | GPU 0: , 205.82 GB VRAM. After training
2026-04-12 14:28:22 | INFO     | __main__ | Training completed. Training result: TrainOutput(global_step=2, training_loss=0.988294243812561, metrics={'train_runtime': 17.9094, 'train_samples_per_second':

In [ ]:
# save
save_adapter(trainer, tokenizer, paths_cfg)
merge_and_save(model_cfg, paths_cfg)

In [ ]:
finish_wandb(
    wandb_run
)
logger.info("Wandb run finished.")